### Задание 1. Парсинг таблицы (0.5 балла)

Извлеките таблицу с курсом валют с сайта Банка России https://www.cbr.ru/currency_base/daily/

Можно собирать теги таблицы через requests и BeautifulSoup циклом (сложнее) **ИЛИ** использовать pandas (удобнее, в одну строку)

In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import numpy as np
import json
import datetime
from time import sleep

In [2]:
# your code here
url = 'https://www.cbr.ru/currency_base/daily/'
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
}
response = requests.get(url, headers=headers)
response.encoding = 'utf-8'  

# Объект Response содержит всю информацию об ответе
print('Тип объекта:', type(response))
print('Статус-код: ', response.status_code)
print('URL запроса:', response.url)
print('Тип содержимого:', response.headers['Content-Type'])

soup = BeautifulSoup(response.text, 'html.parser')
table = soup.find("table")
rows = table.find_all('tr', {'class': 'product_pod'})

headers_row = table.find('tr')
headers = [th.get_text(strip=True) for th in headers_row.find_all('th')]

data_rows = []
for tr in table.find_all('tr')[1:]: 
    cells = tr.find_all('td')
    if cells:
        row = [cell.get_text(strip=True) for cell in cells]
        data_rows.append(row)

data = pd.DataFrame(data_rows, columns=headers)
data.head()


Тип объекта: <class 'requests.models.Response'>
Статус-код:  200
URL запроса: https://www.cbr.ru/currency_base/daily/
Тип содержимого: text/html; charset=utf-8


,Цифр. код,Букв. код,Единиц,Валюта,Курс
0,036,AUD,1,Австралийский доллар,"51,4324"
1,944,AZN,1,Азербайджанский манат,"42,0901"
2,012,DZD,100,Алжирских динаров,"53,7635"
3,051,AMD,100,Армянских драмов,"19,4253"
4,764,THB,10,Батов,"21,9589"


### Задание 2. Парсинг requests + BeautifulSoup (3 балла)

С главной страницы сайта: https://naked-science.ru/ извлеките новости.

1. Передайте ссылку в requests и BeautifulSoup
2. Пройдите циклом по всем тегам 'div' с атрибутами class="news-item grid" и извлеките из каждого:
    - заголовок ('h3')
    - текст превью ('p')
    - автора ('div', class="meta-item_author")
    - ссылку на новость ('a', class="animate-custom") - не забудьте, что сама ссылка хранится в атрибуте 'href', нужно ее извлечь: .get('href')
    - дату (определите тег и атрибут самостоятельно)
    - просмотры (определите тег и атрибут самостоятельно)
    - теги (определите тег и атрибут самостоятельно), их можно собрать из общего тега и предобработать: заменить '#' на пустые кавычки и разделить .split() либо использовать вложенный цикл.
3. Соберите результат в датафрейм

Пример работы вашего цикла:

```
for tag in soup.find_all('div', {'class':"news-item grid"}):
    title = tag.find('h3').text
```

In [12]:
# your code here
# your code here
url = 'https://naked-science.ru/'
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
}
response = requests.get(url, headers=headers)
response.encoding = 'utf-8'  

# Объект Response содержит всю информацию об ответе
print('Тип объекта:', type(response))
print('Статус-код: ', response.status_code)
print('URL запроса:', response.url)
print('Тип содержимого:', response.headers['Content-Type'])

soup = BeautifulSoup(response.text, 'html.parser')
rows = soup.find_all('div', {'class': 'news-item grid'})
headers = ["title", "preview", "author", "new", "data", "views", "tags"]
data_rows = []
for row in rows: 
    title = row.find("h3").text.strip()
    preview = row.find("p").text.strip()
    author = row.find("div", {"class" : "meta-item_author"}).text.strip()
    newUrl = row.find('a', {"class" : "animate-custom"}).get("href")
    data = row.find("span", {"class" : "echo_date"}).text.strip()
    views = row.find("span", {"class" : "fvc-count tetg"}).text.strip()
    tags_line = row.find("div", {"class" : "mobile-one-line-tags"})
    tags = [x.text.strip() for x in tags_line.find_all("a", {"class" : "animate-custom"})]
    data_rows.append([title,preview,author,newUrl,data,views,tags])
data = pd.DataFrame(data_rows, columns=headers)
data.head()


Тип объекта: <class 'requests.models.Response'>
Статус-код:  200
URL запроса: https://naked-science.ru/
Тип содержимого: text/html; charset=utf-8


,title,preview,author,new,data,views,tags
0,Производные кумарина научили бороться с бактер...,Коллектив ученых Томского политехнического уни...,ТПУ,https://naked-science.ru/article/column/proizv...,"2 июня, 07:59",14,"[ТПУ, # антибактериальные препараты, # антибак..."
1,"Российские ученые показали, как меняется связь...",Ученые из Центра вычислительной физики МФТИ вы...,ФизТех,https://naked-science.ru/article/column/svyaz-...,"1 июня, 16:35",81,"[ФизТех, # вязкость, # диффузия, # растворы]"
2,Ученые обнаружили запрещенные погребальные обр...,Законы средневекового Китая требовали хоронить...,Андрей Серегин,https://naked-science.ru/article/archeology/uc...,"1 июня, 14:29",311,"[Археология, # захоронение, # Китай, # обряды]"
3,Орангутаны оказались рекордсменами по длительн...,"Международная группа исследователей выяснила, ...",Татьяна Зайцева,https://naked-science.ru/article/biology/orang...,"1 июня, 14:17",302,"[Биология, # грудное вскармливание, # орангута..."
4,Шум во время пролета объекта мимо Земли удалос...,На высоте более 90 километров над поверхностью...,Адель Романенкова,https://naked-science.ru/article/astronomy/sho...,"1 июня, 11:51","4,2 тыс","[Астрономия, # Звуковые волны, # инфразвук, # ..."


### Задание 3. Вакансии (1.5 балла)

1. Обратитесь к API портала «Работа России» (https://trudvsem.ru/opendata/api) для поиска вакансий.
2. Используйте requests.get(), преобразуйте ответ в json (используйте параметры при запросе: в "text" передайте "Python", "limit" - извлечем 100 записей).
3. Извлеките данные из ответа API. Сохраните собранную информацию в датафрейм.

In [13]:
# your code here
url = "http://opendata.trudvsem.ru/api/v1/vacancies"

params = {
    'text': 'Python',        
    'limit': 100,              
    'offset': 0,              
}

all_vacancies = []
while True:
    response = requests.get(url, params=params)
    data = response.json()
    if response.status_code != 200:
        print(f"Ошибка {response.status_code}: {response.text}")
        break
    vacancies_data = data['results']['vacancies']

    for item in vacancies_data:
        vacancy_info = item.get('vacancy', {})
        company_info = item.get('company', {})
        
        # Извлекаем нужные поля
        salary = vacancy_info.get('salary', {})

        # Формируем запись
        record = {
            'job_name': vacancy_info.get('job-name', 'Нет данных'),
            'region': vacancy_info.get('region', {}).get('name', 'Нет данных'),
            'company_name': company_info.get('name', 'Нет данных'),
            'vacancy_url': vacancy_info.get('vac_url', 'Нет данных'),
            'publish_date': vacancy_info.get('publish_date', 'Нет данных'),
        }
        all_vacancies.append(record)

    # Обновляем offset для следующей страницы
    params['offset'] += params['limit']
    # Небольшая пауза, чтобы не перегружать сервер
    sleep(0.5)
data_vacancies = pd.DataFrame(all_vacancies)
data_vacancies.head()

Ошибка 500: {"status":"500","request":{"api":"v1"},"meta":{"error":"При выполнении запроса произошла ошибка. Проверьте синтаксис запроса и\\или обратитесь в службу поддержки"}}


,job_name,region,company_name,vacancy_url,publish_date
0,Программист Python,Ярославская область,Нет данных,https://trudvsem.ru/vacancy/card/1107608001214...,Нет данных
1,Разработчик Python,Город Москва,Нет данных,https://trudvsem.ru/vacancy/card/5107746005494...,Нет данных
2,Программист Python,Город Москва,Нет данных,https://trudvsem.ru/vacancy/card/1036141003865...,Нет данных
3,Программист Python,Город Москва,Нет данных,https://trudvsem.ru/vacancy/card/1097746819720...,Нет данных
4,Программист Python,Город Москва,Нет данных,https://trudvsem.ru/vacancy/card/1107746455046...,Нет данных


### Задание 4. Землетрясения в мае 2026 (3 балла)

Работаем с API USGS Earthquake Catalog, [документация](https://earthquake.usgs.gov/fdsnws/event/1/)

URL для получения данных: https://earthquake.usgs.gov/fdsnws/event/1/query

Параметры запроса (обязательные):

- format - "geojson"
- starttime – начальная дата (YYYY-MM-DD, возьмем 1 мая 2026)
- endtime – конечная дата (YYYY-MM-DD, возьмем 31 мая 2026)
- minmagnitude – минимальная магнитуда (возьмем 4.5)

Поля (ключи), которые необходимо извлечь из каждого землетрясения:

- mag - Магнитуда
- place - Описание места
- time  - Дата и время события, позже нужно   перевести из миллисекунд в читаемый формат, пример:
```
import datetime

time_ms = 1748419123460
date_obj = datetime.datetime.fromtimestamp(time_ms / 1000)
date_str = date_obj.strftime('%Y-%m-%d')
print(date_str)  # вернет 2025-05-28
```
- felt  - количество людей, почувствовавших землетрясение
- mmi   - интенсивность (измеренная инструментально)
- sig   - интенсивность по показаниям людей
- alert - уровень оповещения (green, yellow, red или None)

**Все перечисленные выше поля извлекаются по ключу `'properties'`**

- longitude, latitude, depth - координаты и глубина (км)

**Извлекаются по ключам '`geometry'` -> `'coordinates'`**, получите список [longitude, latitude, depth]

Чтобы собрать датафрейм из json, нужно:
1. Пройти по **списку словарей, ключ `features`** (это и есть землетрясения) и для каждого извлечь все перечисленные поля.
2. Сложить данные в список словарей или список списков
3. Собрать датафрейм

In [ ]:
# your code here
url = "https://earthquake.usgs.gov/fdsnws/event/1/query"

params = {
    "format" : "geojson",
    "starttime" : "2026-05-01",
    "endtime" : "2026-05-31",
    "minmagnitude" : 4.5
}

response = requests.get(url, params=params)
data = response.json()
records = []

for feature in data['features']:
    props = feature['properties']
    geom = feature['geometry']['coordinates']  
    
    time_ms = props.get('time')
    if time_ms is not None:
        dt = datetime.datetime.fromtimestamp(time_ms / 1000)
        time_str = dt.strftime('%Y-%m-%d %H:%M:%S')
    else:
        time_str = None
    
    record = {
        'mag': props.get('mag'),
        'place': props.get('place'),
        'time': time_str,
        'felt': props.get('felt'),
        'mmi': props.get('mmi'),
        'sig': props.get('sig'),
        'alert': props.get('alert'),
        'longitude': geom[0] if len(geom) > 0 else None,
        'latitude': geom[1] if len(geom) > 1 else None,
        'depth': geom[2] if len(geom) > 2 else None
    }
    records.append(record)

df = pd.DataFrame(records)

print(f"Собрано землетрясений: {len(df)}")
df.head()

Собрано землетрясений: 426


,mag,place,time,felt,mmi,sig,alert,longitude,latitude,depth
0,5.0,"81 km SE of Lae, Papua New Guinea",2026-05-31 03:45:51,NaN,NaN,385,NaN,147.4544,-7.2968,68.854
1,5.6,"137 km SSE of Oistins, Barbados",2026-05-31 02:27:25,22.0,3.194,492,green,-59.1029,11.9084,35.000
2,4.9,"24 km NNW of Murghob, Tajikistan",2026-05-31 02:15:59,NaN,NaN,369,NaN,73.8219,38.3564,131.794
3,4.5,"26 km SE of Ashoro, Japan",2026-05-30 19:34:14,NaN,NaN,312,NaN,143.7857,43.0767,116.722
4,4.6,"63 km W of Catuday, Philippines",2026-05-30 19:21:09,NaN,NaN,326,NaN,119.2212,16.3879,10.000


### Задание 5. Всемирный банк (дополнительно, 2 балла)
Если нет доступа к API Всемирного банка, то:
1) Выберите любой датасет / API
2) Выделите примерную структуру (какие признаки, объем, есть ли ограничения с доступом и т.д.)
3) Ответьте на вопрос: Можно ли обучить модель (регрессия / классификация)?

API Всемирного банка (World Bank Open Data), URL для получения данных: https://api.worldbank.org/v2

Загрузите данные по выбранной стране (можно использовать код страны, например "RU" для России, "US" для США, "CN" для Китая и т.д.) за период с 2000 по 2024 год.

Соберите следующие показатели (коды индикаторов Всемирного банка):

- NY.GDP.MKTP.CD – ВВП (в долларах США)
- NY.GDP.PCAP.CD – ВВП на душу населения
- NY.GDP.MKTP.KD.ZG – Годовой рост ВВП (%)
- FP.CPI.TOTL.ZG – Инфляция (дефлятор ВВП, %)
- SL.UEM.TOTL.NE.ZS – Уровень безработицы (% от общей численности рабочей силы)
- SE.XPD.TOTL.GD.ZS – Расходы на образование (% от ВВП)
- SE.TER.ENRR – Валовой коэффициент охвата высшим образованием (%)
- SP.POP.TOTL – Общая численность населения
- SP.DYN.LE00.IN – Ожидаемая продолжительность жизни при рождении (лет)

Чтобы собрать датафрейм, нужно:
- для каждого индикатора выполнить GET-запрос к API, параметры: "format" - "json", "date" - "2000:2024"
- из ответа API извлечь значения индикатора (ключ "value") за 2000-2024 гг (ключ "date").
- объединить в один датафрейм: строки — годы, столбцы — названия индикаторов (переименуйте для удобства).

**Формат ссылки для запроса**: базовая_ссылка/код_страны/indicator/код_индикатора

Важно:
- API Всемирного банка иногда возвращает пустые значения. Для некоторых показателей данные могут быть не за все годы.
- Запросы лучше выполнять с паузой (не обязательно, но вежливо).

In [ ]:
indicators = ['NY.GDP.MKTP.CD', 'NY.GDP.PCAP.CD', 'NY.GDP.MKTP.KD.ZG', 'FP.CPI.TOTL.ZG', 'SL.UEM.TOTL.NE.ZS', 'SE.XPD.TOTL.GD.ZS', 'SE.TER.ENRR', 'SP.POP.TOTL', 'SP.DYN.LE00.IN']

# your code here
